# Data Mining – Lab 4
## Data Preparation: Data Quality, Cleaning, Integration, Transformation

**Dataset:** StudentsPerformance.csv  

**Лекцийн сэдэв:** Өгөгдөл бэлтгэх ба чанар (номын 2-р бүлэг)


### Lab зорилго
- Өгөгдлийн чанарыг (quality) шалгаж сурах
- Missing values болон noisy data-г цэвэрлэх аргуудыг практикт хэрэглэх
- Өгөгдөл нэгтгэх (integration), давхардал ба хамаарлыг (redundancy) илрүүлэх
- Normalization (min-max, z-score, decimal scaling) ба discretization хийх
- Lab 3-т үзсэн feature engineering-ийг бүрэн дүүрэн data preparation процесс болгон өргөтгөх

## 0) Dataset унших ба ерөнхий шалгалт

Өмнөх lab-уудын адил **StudentsPerformance.csv** dataset-ийг ашиглана.


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("StudentsPerformance.csv")
df.head()

In [ ]:
print(df.shape)
df.info()

## 1) Өгөгдлийн чанар (Data Quality)

Өгөгдөл нь ашиглах зорилгодоо нийцэж байвал чанартай гэж үзнэ.  
Чанарын хүчин зүйлс: **accuracy, completeness, consistency, timeliness, believability, interpretability**.


### TASK 1: Quality check

1. Багана бүрийн дутуу утгын тоог ол (completeness).
2. Давхардсан мөрийн тоог ол.
3. Оноонууд 0–100 мужид байгаа эсэхийг шалга (accuracy).
4. Категори багануудын өвөрмөц утгуудыг харж, бичих хэлбэр зөрүүтэй эсэхийг шалга (consistency).


In [ ]:
missing_per_col = df.isnull().sum()
print("Дутуу утга:\n", missing_per_col)

dup_count = df.duplicated().sum()
print("Давхардсан мөр:", dup_count)

score_cols = ['math score', 'reading score', 'writing score']
print(df[score_cols].agg(['min', 'max']))

for col in ['gender', 'lunch', 'test preparation course']:
    print(col, df[col].unique())

## 2) Data Cleaning – Missing values

StudentsPerformance.csv нь цэвэр dataset тул бодит нөхцөлийг дууриахын тулд  
зарим утгыг **зориудаар дутуу** болгоно (math, reading score-ийн 10%, lunch-ийн 5%).


In [ ]:
rng = np.random.default_rng(42)
df_miss = df.copy()

for col in ['math score', 'reading score']:
    df_miss[col] = df_miss[col].astype(float)
    idx = rng.choice(df_miss.index, size=int(0.10 * len(df_miss)), replace=False)
    df_miss.loc[idx, col] = np.nan

idx = rng.choice(df_miss.index, size=int(0.05 * len(df_miss)), replace=False)
df_miss.loc[idx, 'lunch'] = np.nan

df_miss.isnull().sum()

### TASK 2: Дутуу утгын хувь

Багана бүрийн дутуу утгын **хувийг** ол.


In [ ]:
missing_pct = df_miss.isnull().mean() * 100
missing_pct

### TASK 3: Missing values нөхөх аргууд

Дараах 5 аргыг хэрэглэ:

- a) Дутуу утгатай мөрийг орхих (ignore tuple)
- b) Баганын **mean**-ээр нөхөх
- c) Баганын **median**-аар нөхөх
- d) Ижил **class**-ийн mean-ээр нөхөх (gender-ээр бүлэглэнэ)
- e) Категори баганыг хамгийн их тохиолдох утгаар (**mode**) нөхөх


In [ ]:
# a) Tuple-ийг орхих
df_a = df_miss.dropna()
print("a) үлдсэн мөр:", len(df_a))

# b) Mean
df_b = df_miss.copy()
df_b['math score'] = df_b['math score'].fillna(df_b['math score'].mean())

# c) Median
df_c = df_miss.copy()
df_c['reading score'] = df_c['reading score'].fillna(df_c['reading score'].median())

# d) Ижил class-ийн mean (gender-ээр)
df_d = df_miss.copy()
df_d['math score'] = df_d['math score'].fillna(
    df_d.groupby('gender')['math score'].transform('mean')
)

# e) Mode
df_e = df_miss.copy()
df_e['lunch'] = df_e['lunch'].fillna(df_e['lunch'].mode()[0])

print(df_b['math score'].isnull().sum(),
      df_d['math score'].isnull().sum(),
      df_e['lunch'].isnull().sum())

### TASK 4: Аргуудыг харьцуулах

Анхны (дутуу утгагүй) `math score`-ын mean, std-ийг нөхсөн аргуудтай харьцуул.


In [ ]:
comparison = pd.DataFrame({
    'original': df['math score'].agg(['mean', 'std']),
    'ignore_tuple': df_a['math score'].agg(['mean', 'std']),
    'mean_fill': df_b['math score'].agg(['mean', 'std']),
    'class_mean_fill': df_d['math score'].agg(['mean', 'std'])
})
comparison

### TASK 4.1: Тайлбар (үгээр)

1. Аль арга std-ийг хамгийн ихээр багасгав? Яагаад?
2. Mean-ээр нөхөх нь өгөгдлийг хэрхэн хазайлгаж болох вэ?
3. Хэзээ "tuple орхих" арга зохимжтой вэ?

**Хариулт:**
1. Mean-ээр нөхөх арга std-ийг хамгийн ихээр багасгасан (14.468 → 13.482). Нөхсөн утгууд төв орчимд байрлах тул тархалт нарийсна.
2. Дутуу утгыг дундажаар нөхөхөд хэлбэлзэл хэт бага харагдаж, тархалт төв рүү шахагдана. Үүнээс шинжилгээний дүгнэлт буруу гарч болно.
3. Дутуу утга цөөн (ойролцоогоор 5%-аас бага) бөгөөд санамсаргүй тархсан үед орхих арга зохимжтой.

## 3) Data Cleaning – Noisy data

Noise нь хэмжсэн утга дахь санамсаргүй алдаа эсвэл хэлбэлзэл юм.  
Зөөллөх аргууд: **binning**, **regression**, **outlier analysis**.


### TASK 5: Binning – зөөллөх

Лекцийн жишээ: эрэмбэлсэн үнэ `4, 8, 15, 21, 21, 24, 25, 28, 34`.

1. Өгөгдлийг 3 bin-д (bin бүрт 3 утга) хуваа (equal-frequency).
2. **Bin means**-ээр зөөллө.
3. **Bin boundaries**-аар зөөллө.


In [ ]:
prices = pd.Series([4, 8, 15, 21, 21, 24, 25, 28, 34])

bins = [prices[i:i+3] for i in range(0, len(prices), 3)]

# Smoothing by bin means
smooth_mean = [[round(b.mean())] * len(b) for b in bins]

# Smoothing by bin boundaries
smooth_bound = []
for b in bins:
    lo, hi = b.min(), b.max()
    smooth_bound.append([int(lo) if (v - lo) <= (hi - v) else int(hi) for v in b])

print("Bins      :", [list(b) for b in bins])
print("By means  :", smooth_mean)
print("By bounds :", smooth_bound)

### TASK 6: Outlier илрүүлэх – IQR арга

`math score` дээр IQR аргаар outlier ол.

- Q1 = 25-р percentile, Q3 = 75-р percentile  
- IQR = Q3 − Q1  
- Outlier: `< Q1 − 1.5·IQR` эсвэл `> Q3 + 1.5·IQR`


In [ ]:
q1 = df['math score'].quantile(0.25)
q3 = df['math score'].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outliers = df[(df['math score'] < lower) | (df['math score'] > upper)]
print("lower:", lower, "upper:", upper, "outlier тоо:", len(outliers))
outliers[['math score']].head()

### TASK 7: Outlier илрүүлэх – z-score арга

Mean-аас **2σ**-аас хол утгыг сэжигтэй outlier гэж үзье.


In [ ]:
z = (df['math score'] - df['math score'].mean()) / df['math score'].std()
z_outliers = df[z.abs() > 2]
print("z-score outlier тоо:", len(z_outliers))

### TASK 7.1: Тайлбар (үгээр)

1. IQR болон z-score аргаар олдсон outlier-ийн тоо ялгаатай байна уу? Яагаад?
2. Оноо 0 байх нь "алдаа" юм уу, эсвэл бодит утга уу? Outlier-ийг устгахаас өмнө юуг шалгах ёстой вэ?

**Хариулт:**
1. Тоо ялгаатай: IQR-аар 8, z-score-оор 43. z-score нь mean, std-д тулгуурладаг тул outlier өөрөө тэдгээрийг нөлөөлдөг. Мөн оноо 100-аас дээш байж чадахгүй тул z-score дээд талд хэт олон утгыг буруу тэмдэглэдэг.
2. Оноо 0 нь алдаа биш бодит утга гэж үзэж байна: тухайн оюутны reading 17, writing 10 байгаа нь бүх хичээлд сул гүйцэтгэлтэй байгааг харуулна. Устгахаас өмнө эх сурвалж, бусад баганын утгыг шалгах ёстой.

## 4) Data Integration – Өгөгдөл нэгтгэх

Олон эх сурвалжийн өгөгдлийг нэг санд нэгтгэхэд **entity identification**, **redundancy**,  
**tuple duplication**, **data value conflict** асуудлууд гарна.  

Бодит нөхцөлийг дууриахын тулд dataset-ийг хоёр хүснэгтэд хувааж, давхардал нэмнэ.


In [ ]:
students = df[['gender', 'race/ethnicity', 'parental level of education',
               'lunch', 'test preparation course']].copy()
students.insert(0, 'student_id', range(1, len(df) + 1))

scores = df[['math score', 'reading score', 'writing score']].copy()
scores.insert(0, 'student_id', range(1, len(df) + 1))

# Давхардсан 5 мөр нэмнэ
scores_dup = pd.concat([scores, scores.sample(5, random_state=1)], ignore_index=True)
print(len(scores), "->", len(scores_dup))

### TASK 8: Давхардал устгах ба нэгтгэх

1. `scores_dup` дахь давхардсан `student_id`-тай мөрийг устга.
2. `students` ба `scores_clean` хүснэгтийг `student_id` дээр нэгтгэ (inner join).


In [ ]:
scores_clean = scores_dup.drop_duplicates(subset=['student_id'])
merged = students.merge(scores_clean, on='student_id', how='inner')

print(len(scores_clean), merged.shape)
merged.head()

### TASK 9: Redundancy – Correlation analysis

Тоон attribute-уудын хоорондын корреляцийг ол. Өндөр корреляцтай attribute нь илүүдэл (redundant) байж болно.


In [ ]:
corr = merged[['math score', 'reading score', 'writing score']].corr()
corr

### TASK 10: Nominal attribute – Chi-square тест

`gender` ба `lunch` хоёр хамааралтай эсэхийг χ² тестээр шалга.  
(p-value < 0.05 бол хамааралтай гэж үзнэ.)


In [ ]:
from scipy.stats import chi2_contingency

contingency = pd.crosstab(merged['gender'], merged['lunch'])
chi2, p, dof, expected = chi2_contingency(contingency)

print("chi2 =", round(chi2, 3), " p-value =", round(p, 4), " dof =", dof)
contingency

### TASK 10.1: Тайлбар (үгээр)

1. Аль хоёр оноо хамгийн өндөр корреляцтай вэ? Тэдгээрийн нэгийг хасаж болох уу?
2. Chi-square тестийн үр дүнгээс юу дүгнэж байна вэ?

**Хариулт:**
1. Reading ба writing хооронд корреляц хамгийн өндөр (0.819). Бүрэн давхардал гэх хэмжээ биш тул хоёуланг нь хадгалсан нь зүйтэй.
2. χ² = 1.63, p = 0.202 > 0.05 тул gender ба lunch хооронд статистикийн хувьд мэдэгдэхүйц хамаарал байхгүй гэж дүгнэв.

## 5) Data Transformation – Normalization

Хэмжих нэгж, муж өөр attribute-ууд шинжилгээнд өөр жин авдаг.  
Normalization нь бүх attribute-д ижил жин өгөхийг зорино.


### TASK 11: Гурван normalization арга

`math score` дээр дараах аргуудыг хэрэглэ:

- **Min-Max:** `(v − min) / (max − min)`
- **Z-score:** `(v − mean) / std`
- **Decimal scaling:** `v / 10^j` (j нь max(|v'|) < 1 байх хамгийн бага бүхэл тоо)


In [ ]:
x = df['math score']

df['math_minmax'] = (x - x.min()) / (x.max() - x.min())
df['math_z'] = (x - x.mean()) / x.std()

j = len(str(int(x.abs().max())))
df['math_decimal'] = x / 10 ** j

df[['math score', 'math_minmax', 'math_z', 'math_decimal']].head()

### TASK 12: Харьцуулалт

Дөрвөн баганын min, max, mean, std утгыг нэг хүснэгтээр харуул.


In [ ]:
cols = ['math score', 'math_minmax', 'math_z', 'math_decimal']
df[cols].describe().loc[['min', 'max', 'mean', 'std']]

### TASK 12.1: Тайлбар (үгээр)

1. Аргуудын гаралтын муж хэд вэ?
2. Outlier олонтой үед аль арга илүү тохиромжтой вэ? Яагаад?
3. Шинэ өгөгдөл ирэхэд яагаад min, max (эсвэл mean, std)-ийг хадгалах хэрэгтэй вэ?

**Хариулт:**
1. Min-max: [0, 1]; z-score: ойролцоогоор [−4.85, 2.07]; decimal scaling: [0, 0.1].
2. Outlier олонтой үед z-score илүү тохиромжтой, учир нь min-max-ийн муж хэт утгаар тодорхойлогддог.
3. Шинэ өгөгдөлд ижил масштаб ашиглахын тулд сургалтын өгөгдлийн min, max (эсвэл mean, std)-ийг хадгалах хэрэгтэй.

## 6) Data Transformation – Discretization

Тоон утгыг интервал эсвэл ойлголтын шошгоор (Low, Medium, High) солино.


### TASK 13: Equal-width

`math score`-ыг **ижил өргөнтэй** 3 интервалд хуваа.


In [ ]:
df['math_width'] = pd.cut(df['math score'], bins=3, labels=['Low', 'Medium', 'High'])
df['math_width'].value_counts().sort_index()

### TASK 14: Equal-frequency

`math score`-ыг **ойролцоо тооны утгатай** 3 бүлэгт хуваа.


In [ ]:
df['math_freq'] = pd.qcut(df['math score'], q=3, labels=['Low', 'Medium', 'High'])
df['math_freq'].value_counts().sort_index()

### TASK 15: Concept hierarchy (өөрийн босго)

Lab 3-ын ангиллыг ашигла:  
`< 60` → Low, `60–79` → Medium, `≥ 80` → High


In [ ]:
df['math_level'] = pd.cut(df['math score'], bins=[0, 60, 80, 101],
                          labels=['Low', 'Medium', 'High'], right=False)
df['math_level'].value_counts().sort_index()

### TASK 15.1: Тайлбар (үгээр)

1. Equal-width ба equal-frequency аргын bucket бүрийн тоо ямар ялгаатай байна вэ?
2. Аль арга outlier-т илүү мэдрэмтгий вэ?
3. Бодит хэрэглээнд аль аргыг хэзээ сонгох вэ?

**Хариулт:**
1. Equal-width: 8 / 387 / 605; equal-frequency: 348 / 344 / 308. Equal-width-д bucket-ийн тоо эрс ялгаатай байна.
2. Equal-width илүү мэдрэмтгий, учир нь хязгаарыг min, max тодорхойлдог.
3. Тоон утга жигд тархсан, интервалын өргөн чухал бол equal-width; bucket бүрд ойролцоо тооны утга хэрэгтэй бол equal-frequency; бодит шатлал байвал concept hierarchy сонгоно.

## 7) Бүх алхмын дараалал (Concept Check)

Өгөгдөл бэлтгэх процесс: **чанар шалгах → цэвэрлэх → нэгтгэх → хувиргах**.


### TASK 16: Pipeline thinking (Verbal)

1. Дээрх 4 алхмыг яагаад энэ дарааллаар хийдэг вэ?
2. Normalization-ийг missing values нөхөхөөс ӨМНӨ хийвэл ямар асуудал гарах вэ?
3. Классификацийн lab-д орохоос өмнө өөрийн dataset дээр аль алхмуудыг заавал хийх вэ?

**Хариулт:**
1. Чанар шалгах → цэвэрлэх → нэгтгэх → хувиргах. Цэвэрлэхээс өмнө нэгтгэвэл давхардал, дутуу утга тархана. Хувиргалт нь цэвэрлэсэн өгөгдөл дээр хийгдэх ёстой.
2. Дутуу утгыг нөхөхөөс өмнө normalization хийвэл mean, std зэрэг параметр буруу тооцогдож, нөхсөн утгууд буруу масштабтай болно.
3. Mean/median нөхөлт, outlier шалгалт, давхардал устгах, нэгтгэх, normalization болон discretization хийх, мөн параметрүүдээ хадгалах.

### Тайлбар
- 


# Data Mining – Lab 4 Тайлан
## Өгөгдөл бэлтгэх ба чанар

**Нэр:** А.Сүхболд  
**Оюутны код:** s22c055b  
**Анги / Бүлэг:**  
**Огноо:** 2026 он 10 сар 14 өдөр

---

## 1. Lab зорилго
Өгөгдлийн чанарыг шалгаж, дутуу болон алдаатай байж болзошгүй утгуудыг боловсруулан өгөгдлийг нэгтгэх, normalization болон discretization хийх. Эдгээр аргууд өгөгдөлд хэрхэн нөлөөлж байгааг StudentsPerformance өгөгдөл дээр харьцуулан судлав.

---

## 2. Dataset-ийн товч тайлбар
- **Dataset нэр:** StudentsPerformance.csv
- **Мөр, баганын тоо:** 1000 мөр, 8 багана
- **Ашигласан үндсэн баганууд:** math score, reading score, writing score (тоон); gender, race/ethnicity, parental level of education, lunch, test preparation course (ангилал)

---

## 3. Data Quality ба Cleaning

### 3.1 Missing values
- **Ямар аргуудыг хэрэглэсэн бэ:** Дутуу мөрийг орхих, mean, median, ижил class-ийн mean, mode.
- **Аль нь тохиромжтой вэ:** Мөр орхих арга 228 мөр (22.8%) алдсан тул энэ өгөгдөлд тохирохгүй. Mean ба class mean нь мөрийг хадгалдаг ч std-ийг бууруулдаг тул нөхөхдөө анхаарах хэрэгтэй.

### 3.2 Noisy data
- **Binning:** Bin means-ээр [9, 9, 9], [22, 22, 22], [29, 29, 29]; bin boundaries-аар [4, 4, 15], [21, 21, 24], [25, 25, 34] болсон.
- **Outlier:** IQR аргаар 8, z-score аргаар 43 outlier олдсон.

---

## 4. Data Integration
- Оюутны мэдээлэл болон шалгалтын оноог хоёр хүснэгтэд хуваасан. Онооны хүснэгтэд 5 мөр давхардуулж, `student_id`-аар давхардлыг устгаад inner join хийсэн. Үр дүн 1000 × 9.
- **Correlation:** reading–writing 0.819 (хамгийн өндөр), math–reading 0.735, math–writing 0.733.
- **Chi-square:** chi^2 = 1.63, p = 0.202 > 0.05. Gender ба lunch хооронд мэдэгдэхүйц хамаарал алга.

---

## 5. Data Transformation

### 5.1 Normalization
- Min-max, z-score, decimal scaling (j = 3) аргуудын гаралтын муж тус тусдаа [0, 1], mean 0 std 1, [0, 0.1] байна.
- Outlier олон үед z-score илүү тохиромжтой. Шинэ өгөгдөлд хэрэглэхийн тулд сургалтын өгөгдлийн параметрүүдийг хадгалах хэрэгтэй.

### 5.2 Discretization
- Equal-width: 8 / 387 / 605. Equal-frequency: 348 / 344 / 308. Concept hierarchy (<60, 60–79, ≥80): 228 / 510 / 262.
- Equal-width нь хязгаарыг min, max тодорхойлдог тул outlier-т илүү мэдрэмтгий.

---

## 6. Сэтгэхүй
Өгөгдөл бэлтгэх дараалал нь чанар шалгах → цэвэрлэх → нэгтгэх → хувиргах. Цэвэрлэхээс өмнө нэгтгэвэл давхардал, дутуу утга тархана. Normalization-ийг дутуу утга нөхөхөөс өмнө хийвэл mean, std буруу гарна. Тиймээс энэ дарааллыг баримтлах нь зүйтэй.

---

## 7. Reflection
1. Өгөгдлийн чанар муу байвал data mining-ийн үр дүн буруу, итгэлгүй болно.
2. Дутуу утга нөхөгдсөн, outlier шалгагдсан, давхардал арилгасан, normalization хийсэн өгөгдөл дараагийн classification lab-д бэлэн.
3. *(Ойлгомжгүй хэсгээ оюутан өөрөө бичнэ.)*

---

## 8. Дүгнэлт
Өгөгдөл бэлтгэх алхам бүр үр дүнд өөрөөр нөлөөлдөг. Дутуу утгыг нөхөх арга, outlier-ийн шалгуур, discretization-ийн аргыг зөв сонгох нь шинжилгээний найдвартай байдалд шууд нөлөөлнө. Энэ lab-оор өгөгдөл бэлтгэх үндсэн аргуудыг бодит өгөгдөл дээр туршиж, үр дүнг харьцуулж сурлаа.